# 🚗 The DeLorean Engine
## Small can be powerful

**Section 3** of *Search, Extract & Value: The Lightning Fast Assistant* 🚗💨

---

> ### 🖥️ Local only — and that is the entire point.
> There is no `_api` twin of this notebook. Every other section ships one so
> you can pick; this one is about the models you run on your own hardware.

### What you need

| | |
|---|---|
| 🎮 **A GPU** | **under 2 GB of VRAM.** Less than Section 2 needed. |
| 🌐 **Internet** | only the first time, to download 5.7 GB |
| ⏱️ **Time** | about two minutes |

---

Here is the thing nobody mentioned for the last hour.

**Every demo you have seen ran on a small model, on one graphics card, in this
room.** Not a data centre. Not an API. Not a subscription.

| Section | Model | On the card |
|---|---|---|
| 2 — The Flux Capacitor | Qwen3.5 **0.8B** | 1.4 GB |
| 3, 4, 6, 7, 8 | Qwen3.5 **4B** | 2.95 GB |

No question left the building. No question cost anything. The parcel data, the
policy manual, the model and the map were all on one machine.

**And in Section 9 we measured something better than that.** The same ratio
study, asked of the 4B model on a laptop and of a large hosted model in the
cloud, came back with *byte-identical* numbers. Not close — identical. Because
`ratio_study` did the statistics either way. The model's job was to decide
what to ask, not to be clever about arithmetic.

> ### That is why a small model is enough: it is not doing the hard part.

This notebook proves it is not a fluke, by running Section 9 again on a
**different model that is smaller still** — and by showing you the one that
did not work, and why.

---

# 🚦 LIVE DEMO PATH

## Setup

**One line is different from `09_88mph_local.ipynb`.** It is the model name.
Everything else — the tools, the prompt, the graph — is imported from
`src/delorean/`, unchanged.

> ### ⏭️ One thing about this notebook
> It is §3, but **Acts 2 to 5 travel forward to §9** and re-run that section's
> ratio study on a small model. They are clearly fenced off, and the live §3
> path skips straight over them. See the spoiler divider below.


In [1]:
from delorean import agent, data, llm, models, prompts, tools, viz
from delorean.tools import review

#            vvvvvv   this is the whole change
model, tokenizer = llm.load_local("smol")
respond = agent.local_responder(model, tokenizer, max_new_tokens=420)

data.check()

Loading SmolLM3 3B onto NVIDIA GeForce RTX 3080 ...


W0825 12:24:39.178000 26128 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Loading weights:   0%|          | 0/326 [00:00<?, ?it/s]

Ready. Using 1.89 GB of 10.0 GB (4-bit NF4).


---

## Act 1 — 🐭 What did that actually cost?

Before asking it anything, look at what is now sitting on the graphics card.

In [2]:
where = llm.where_it_runs(model)

print(f"  running on : {where['device_name']}")
print(f"  weights    : {where['on_gpu']} of {where['on_gpu'] + where['on_cpu']}"
      f" tensors on the GPU")
print(f"  using      : {where['allocated_gb']:.2f} GB")
print(f"  still free : {where['free_gb']:.1f} GB of {where['total_gb']:.0f} GB")

  running on : NVIDIA GeForce RTX 3080
  weights    : 326 of 326 tensors on the GPU
  using      : 1.85 GB
  still free : 6.9 GB of 10 GB


Under two gigabytes.

For scale: that is smaller than a phone's worth of photos, and **it leaves
enough room on an 8 GB card to run three more of them side by side.** The card
in question is a consumer graphics card, not server hardware.

---

# ⏭️ SPOILER ALERT — this part travels forward to §9

**Everything from here to Act 6 jumps ahead to
[§9 Reaching 88 MPH](09_88mph_local.ipynb).** It takes that section's
ratio-study agent and runs it on a smaller engine, so it will make a great deal
more sense once you have actually been there.

That is deliberate. You are holding a time machine — occasionally it shows you
the future.

> ### 🚦 Reading in order?
> **Skip down to ⏪ BACK TO 1985** and come back here after §9. Nothing in §4
> onwards depends on these four acts.

> 🎬 *"I'm sure that in 1985 plutonium is available in every corner drugstore,
> but in 1955 it's a little hard to come by."* — Doc Brown


---

## Act 2 — 🏎️ Section 9, again, on a smaller engine

Same three tools. Same system prompt. Same graph. Same questions.

If small models were a compromise, this is where it would show.

In [3]:
assistant = agent.build_agent(respond,
                              ["run_sql", "ratio_study", "map_parcels"],
                              max_steps=5)

hoods = agent.ask(assistant,
                  "Run the assessment ratio study broken down by neighborhood.",
                  system=prompts.REVIEW_ASSISTANT)

print(agent.final_answer(hoods))
print()

for step in hoods["trace"]:
    if step["step"] == "act":
        print("SQL it wrote for itself:")
        print("   ", step["args"].get("sql"))

for step in hoods["trace"]:
    if step["step"] == "act" and step["tool"] == "ratio_study":
        viz.show_ratio_table(step["result"]["data"]["results"],
                             title="Found by a 3B model, on 1.85 GB")

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


The ratio study by neighborhood shows that the assessed values are not consistently fair across all neighborhoods. Courthouse Square fails the Uniformity test, Hilldale fails the Equity test, and Riverside Heights fails the Level test. The study highlights areas where the assessed values are either too high or too low compared to the sales prices, indicating potential equity issues.

SQL it wrote for itself:
    SELECT parcel_id FROM sales_feed WHERE assessed_value > 0 ORDER BY neighborhood ASC


neighborhood,sales,Median ratio,COD,PRD,verdict
Courthouse Square,30,0.926,✗ 22.08,1.028,✗ FAIL
Hilldale,50,0.921,13.63,✗ 1.043,✗ FAIL
Riverside Heights,32,✗ 0.809,6.21,1.001,✗ FAIL
Eastwood Ravine,27,0.984,6.06,1.001,✓ PASS
Lyon Estates,42,0.948,6.04,1.001,✓ PASS
Twin Pines Ranch,29,0.929,9.33,1.002,✓ PASS


### 👀 Three reds, in three different columns

That is the same finding Section 9 produced — Courthouse Square failing on
uniformity, Hilldale on equity, Riverside Heights on level, and the county as
a whole passing everything.

**The numbers are not similar. They are the same numbers.** They have to be:
`ratio_study` computed them, and it does not know or care which model asked.

And the map still works.

In [4]:
mapped = agent.ask(assistant,
                   "Show me the sales in Riverside Heights on a map.",
                   system=prompts.REVIEW_ASSISTANT)

print(agent.final_answer(mapped))

viz.show_parcel_map(review.last_map(), context=data.parcels_df(),
                    title="Riverside Heights, drawn by a 3B model")

The map shows 36 parcels in Riverside Heights, colored by their assessment ratio. 30 are assessed below the IAAO range (0.90 to 1.10), and 3 are above it. The map highlights areas where properties are over- or under-assessed.


---

## Act 3 — 🔍 Where the small model slips, and how you catch it

Two for two. Now the third question, and this one does **not** go well — which
is more useful than if it had.

In [5]:
county = agent.ask(assistant,
                   "How did the county do on the ratio study this year?",
                   system=prompts.REVIEW_ASSISTANT)

print(agent.final_answer(county))
print()
for step in county["trace"]:
    if step["step"] == "act":
        print("What it actually asked the tool for:")
        print("   ", step["args"].get("sql"))
        print("   group_by:", step["args"].get("group_by") or "(none)")

The ratio study this year met the IAAO standards for the Eastwood Ravine neighborhood, with a median ratio of 0.9909, COD of 6.172, and PRD of 0.9982. However, the Riverside Heights neighborhood did not have enough sales to conduct a full study. The overall assessment of the county's ratio study this year is a PASS, with all 2 neighborhoods meeting the IAAO standards.

What it actually asked the tool for:
    SELECT parcel_id FROM sales_feed WHERE year_built BETWEEN 2010 AND 2020 AND assessed_value > 0 AND arms_length = 1 AND neighborhood IN ('Hilldale', 'Lyon Estates', 'Riverside Heights', 'Eastwood Ravine', 'Courthouse Square', 'Twin Pines Ranch') AND year_built > 2000 AND assessed_value > 0
   group_by: neighborhood


### Read the query, not the answer

The question was *"how did the **county** do"*. Look at what it asked for:

```sql
... WHERE year_built BETWEEN 2010 AND 2020 AND ...
```

**Nobody asked for a date range.** It invented one, studied only homes built
in that decade, and most neighborhoods had too few qualifying sales to judge —
so it reported back on the two that survived and called it the county.

Now notice what did **not** happen.

- The tool did not get it wrong. *"All 2 neighborhoods meet the IAAO
  standards"* is a perfectly true statement about the rows it was handed.
- Nothing was fabricated. The medians and CODs it quoted are real.
- **The mistake is sitting in plain sight in the SQL**, where anybody can read
  it in about four seconds.

> ### The tool guarantees the arithmetic. It cannot guarantee the model asked for the right rows.

This is the honest shape of the trade-off, and it is worth carrying back to
the office more than any benchmark score:

| | Who is responsible | How you check it |
|---|---|---|
| The **numbers** | the tool — the same code every time | it is arithmetic; test it once |
| The **selection** | the model — and it can wander | **read the query it wrote** |
| The **wording** | the model — a summary, not a source | read it against the table |

The 4B model wrote `SELECT parcel_id FROM sales_feed` for this question —
plain, unfiltered, correct. That difference is what you are buying with the
extra gigabyte, and it is much narrower than "big model good, small model
bad".

🎤 **This is also why Section 9 puts the SQL on screen.** Not to show off —
so that a mistake like this one is visible instead of buried.

---

## Act 4 — 📊 Side by side

Both of these were measured in this repository, on the same card, running the
same ten questions with the same three tools.

**Recall** = did it reach for the right tool when it needed one.
**Restraint** = did it answer directly when it should have, instead of
hitting the database because somebody said "good morning".

In [6]:
rows = [
    ("Qwen3.5 4B", "qwen", "6/6", "4/4", "~10s"),
    ("SmolLM3 3B", "smol", "6/6", "4/4", "6.7s"),
]

print(f"  {'Model':<16}{'Params':>8}{'VRAM':>9}{'Recall':>9}"
      f"{'Restraint':>11}{'Speed':>8}")
print("  " + "-" * 61)
for label, key, recall, restraint, speed in rows:
    spec = models.get(key)
    vram = f"{spec.measured_vram_gb:.2f} GB" if spec.measured_vram_gb else "—"
    print(f"  {label:<16}{spec.params:>8}{vram:>9}{recall:>9}"
          f"{restraint:>11}{speed:>8}")

  Model             Params     VRAM   Recall  Restraint   Speed
  -------------------------------------------------------------
  Qwen3.5 4B            4B  4.00 GB      6/6        4/4    ~10s
  SmolLM3 3B            3B  2.77 GB      6/6        4/4    6.7s


Both models reached for the right tool every time, and both left the tools
alone on every question that did not need one.

⚠️ **But read that table for what it does not say.** "Recall 6/6" means it
picked the correct tool — it says nothing about whether the *arguments* were
sensible, and Act 3 is exactly that gap: SmolLM3 called `ratio_study`
correctly and handed it the wrong rows. Our evaluation scores tool choice
because tool choice is easy to score. **The harder question is whether the
query was right, and the only reliable check is a human reading it.**

So: not "the small model is just as good". Rather —

> ### The small model is enough for the part it is doing, and the part it is doing is smaller than you think.

---

## Act 5 — ⚠️ It scored zero the first time

This is the part to take back to the office, because it cost an afternoon and
it would have cost you one too.

On the first run, SmolLM3 scored **0 out of 6**. It never called a single
tool. The obvious conclusion was that a 3B model simply is not up to it.

That was wrong. It was calling tools perfectly — **we were not listening.**

A model does not "call a tool". It writes text, and something on our side
notices. Qwen writes XML wrapped in `<tool_call>` tags. SmolLM3's own
instructions tell it to write JSON in those same tags. What SmolLM3 actually
writes is the JSON **with no tags at all** — and Gemma 4 brings its own
delimiters again, which is a third convention for the same idea:

In [7]:
qwen_style = (
    "<tool_call>\n<function=run_sql>\n<parameter=sql>\n"
    "SELECT owner_name FROM parcels\n</parameter>\n</function>\n</tool_call>")

smol_style = ('{"name": "run_sql", "arguments": '
              '{"sql": "SELECT owner_name FROM parcels"}}')

# Gemma 4 fences its strings with <|"|> instead of quoting them, which is what
# lets a whole SQL statement -- commas, quotes, braces -- sit inside the braces.
gemma_style = ('<|tool_call>call:run_sql{sql:<|"|>'
               'SELECT owner_name FROM parcels<|"|>}<tool_call|>')

for label, text in [("Qwen3.5 / Nemotron", qwen_style),
                    ("SmolLM3 & Phi-4-mini", smol_style),
                    ("Gemma 4", gemma_style)]:
    calls = tools.parse_tool_calls(text)
    print(f"  {label:<22} -> {[(c.name, list(c.args)) for c in calls]}")

print()
print("  and it does not fire on ordinary prose:")
for text in ['The set {1, 2} is not a tool call.',
             '{"name": "launch_missiles", "arguments": {"x": 1}}']:
    print(f"     {text[:46]:<48} -> {tools.parse_tool_calls(text) or 'no call'}")

  Qwen3.5 / Nemotron     -> [('run_sql', ['sql'])]
  SmolLM3 & Phi-4-mini   -> [('run_sql', ['sql'])]
  Gemma 4                -> [('run_sql', ['sql'])]

  and it does not fire on ordinary prose:
     The set {1, 2} is not a tool call.               -> no call
     {"name": "launch_missiles", "arguments": {"x":   -> no call


### 🎯 The lesson, and it is the most useful thing in this notebook

We had checked the tool format by **rendering each model's chat template** and
reading what it asked for. That felt rigorous. It was not.

> ### A chat template tells you what a model was *trained* to emit. It does not tell you what the model *actually* emits.

The only way to know is to generate from the model and read the raw output —
which is exactly what Section 7 put on screen, and why that beat is worth
keeping. Three lines of parsing in
[`tools/base.py`](../src/delorean/tools/base.py) turned this model from
useless to perfect.

**And then it happened twice more.** Adding Phi-4-mini and Gemma 4 to this
repository turned up the same failure wearing two new disguises:

| Model | Looked like | Actually was |
|---|---|---|
| SmolLM3 | a 3B that cannot use tools — **0/6** | writing perfect calls **without the tags** our parser required |
| Phi-4-mini | echoes tool definitions back, calls nothing | handed a tool-list **shape it did not recognise**, and a template that **silently dropped the tools entirely** |
| Gemma 4 | never calls a tool — **0/6** | its delimiters are **special tokens**, and our decoder was deleting them before the parser ever saw them |

Gemma's is the sharpest, because the whole difference was one keyword
argument — `skip_special_tokens=True` — turning a 6/6 into a 0/6 on identical
generations. After the fix it scores **5/6** and is the fastest model here.

**If you evaluate a model for your office and it scores zero, suspect your
plumbing before you blame the model.** Print the raw output with `repr()`
before you believe any scoreboard, including this one.

---

# ⏪ BACK TO 1985 — the questions that matter on Monday

**You are back in §3.** Nothing below needs anything from §9. It is about what
actually fits on the hardware you already own, and how to choose an engine for
your own office.


---

## Act 6 — 🚫 The one that did not work

Honesty is more useful than a clean sweep, so: the third model in our registry
**cannot do this job on a Windows machine**, and finding out why is the other
half of "how do I choose a model".

`nvidia/NVIDIA-Nemotron-3-Nano-4B` is a well-regarded model that scored 95% on
the same tool-calling evaluation Qwen won. It loads here in 3 GB without
complaint. Then the first question asks for **21 GB** and dies.

It is a **hybrid Mamba** model rather than a plain transformer. That
architecture is fast and memory-efficient — *when* its custom CUDA kernels are
installed. Without `mamba-ssm`, `causal-conv1d` and `triton`, none of which
install cleanly on Windows, it falls back to a plain PyTorch implementation
that costs about **12 MB of VRAM for every token of the prompt**. A normal
transformer costs roughly a hundredth of that.

In [8]:
# Measured on a 10 GB card. Not re-run here -- the last two rows crash.
measured = [(256, 6.25), (512, 9.35), (768, 12.47), (1024, 15.57),
            (1280, None), (1600, None)]

print(f"  {'prompt tokens':>14}   {'peak VRAM':>10}")
print("  " + "-" * 34)
for tokens, peak in measured:
    shown = f"{peak:.2f} GB" if peak else "out of memory"
    print(f"  {tokens:>14}   {shown:>13}")

print()
print("  Section 9's prompt, with three tools attached:")
print(f"     {len(tokenizer(prompts.REVIEW_ASSISTANT)['input_ids']):,} tokens "
      f"of system prompt alone, ~1,600 with the tool schemas.")
print("  So it never gets as far as answering.")

   prompt tokens    peak VRAM
  ----------------------------------
             256         6.25 GB
             512         9.35 GB
             768        12.47 GB
            1024        15.57 GB
            1280   out of memory
            1600   out of memory

  Section 9's prompt, with three tools attached:
     185 tokens of system prompt alone, ~1,600 with the tool schemas.
  So it never gets as far as answering.


⚠️ **A wrong diagnosis, kept on purpose.** The first explanation was that
Nemotron advertises a 262,144-token context and transformers was preallocating
a key-value cache for all of it. The arithmetic was persuasive —
`262144 × 1024 × 42 layers × 2` comes to almost exactly 21 GB.

Capping the context to 8,192 changed nothing. Same failure, same byte count,
to the byte.

> **A number that matches your theory is not evidence that your theory is
> right.** Change the thing you think is responsible and see whether the
> failure moves. This one did not.

None of this makes Nemotron a bad model. On Linux with its kernels installed
it is reportedly excellent. It makes it **the wrong model for this machine**,
which is a different and much more practical thing to know.

---

## Act 7 — 🧭 Choosing one for your office

The registry prints everything this repo has measured, so you can decide
rather than guess.

In [9]:
models.compare()


Qwen3.5 0.8B  (0.8B, fp16)
  Qwen/Qwen3.5-0.8B
  License : Apache 2.0
  VRAM    : 1.44 GB measured   Speed: ~14 tokens/sec
  Good at :
      + Starting up in about three seconds
      + Running on a laptop with no GPU at all
      + Showing how generation works, one token at a time
  Watch   :
      - Not for agent work — it is here to be understood, not to reason
      - Will confidently get facts wrong

Qwen3.5 4B  (4B, 4-bit NF4)
  Qwen/Qwen3.5-4B
  License : Apache 2.0
  VRAM    : 4.0 GB measured   Speed: 8.7s per agent question (Section 9 toolbelt)
  Good at :
      + Tool calling — ranked 1st of 13 local models in a 2026 evaluation
      + Long documents (262,144 token context)
      + Punching far above its size; it beat models 5-7x larger
  Watch   :
      - Eager to call several tools at once when one at a time would do
      - Thinking mode is ON by default and adds a pause before answering

SmolLM3 3B  (3B, 4-bit NF4)
  HuggingFaceTB/SmolLM3-3B
  License : Apache 2.0
  VRAM

---

### 🔒 "I have never heard of these companies"

It is a fair question and you will be asked it, probably by someone who has to
justify this to a commission. Two answers, and the second is the real one.

**The easy answer:** two of the models above are from **Microsoft** and
**Google**. If starting from a name your IT department already works with makes
the first conversation easier, start there — `phi` and `gemma` are one word
each, and both scored 5/6 on the same test Qwen passed 6/6.

**The better answer is that it does not matter.** A model is a *file of
numbers* sitting on your disk. It makes no network calls. Pull the ethernet
cable out of this machine and every demo in this notebook keeps working — the
weights cannot phone home, because there is no code in them to phone home
*with*. Weights do not execute. Whoever trained the file has no more reach into
your county than the author of a PDF you downloaded last week.

What genuinely deserves an audit is the **code around** the model — the tools,
the queries, what it is allowed to touch. That is this repository, and every
line of it is here for you to read.

> ### 🎯 "Where was this model trained?" is a reasonable question. "What can this software reach?" is the one that protects you — and the answer here is: nothing but the folder it sits in.

### The four questions worth asking, in order

1. **Does it run on the hardware you actually have?** Not the hardware in the
   benchmark. Check the architecture has working kernels on *your* operating
   system — that is what Nemotron cost us, and no leaderboard mentions it.
2. **Can it call tools reliably?** This is the whole ballgame for an office
   assistant, and it is not the same skill as writing good prose. Measure it
   on *your* tools, with `scripts/eval_tools.py` as a starting point.
3. **What does the licence let you do?** Four of these are Apache 2.0, one is
   MIT, and Nemotron's is NVIDIA's own. Every one is *ungated* — no form to
   sign before downloading — which is not true of every model out there, and
   was not true of Gemma 3. If this is going near public records, somebody in
   your organisation will need to read it.
4. **Is it small enough to be boring?** A model that fits in 2 GB can run on a
   workstation nobody had to buy, restart in ten seconds, and be replaced next
   year when something better appears.

> ### 🎯 Do not shop for the smartest model. Shop for the smallest one that passes your own test — because the tools are doing the part that has to be right.

---

## 🎬 That's Section 3

1. **Every demo in this talk ran on a small local model.** 0.8B for Section 2,
   4B for everything after it, never more than 3 GB.
2. **A 3B model runs the Section 9 review** — same tools, same prompt, same
   graph, one line changed — on 1.85 GB, faster than the 4B.
3. **The numbers are identical** across models, because the tools compute them
   and the model only decides what to ask.
4. **It slipped once, and the slip was visible.** It invented a date filter
   nobody asked for. The tool answered honestly about the rows it was given,
   and the mistake was legible in the query — which is the argument for
   putting the query on screen.
5. **It scored 0/6 until we fixed our own parser.** A chat template tells you
   what a model was trained to emit, not what it emits.
6. **One model did not work at all here**, for a reason no benchmark reports:
   its architecture needs CUDA kernels that Windows will not give it.

> ### 🚗 "It's not the size of the engine. It's whether the flux capacitor is wired up right."

**Next:** Section 10 — where the assistant stops being limited to one office's
database. 🚗💨

---

# 🔬 EXPLORE ON YOUR OWN

## 7. 🔁 Run it against every model you have

`scripts/eval_tools.py` scores a model on the tools in this repo. Point it at
whichever ones you have downloaded and compare — on your questions, not
somebody else's benchmark.

In [10]:
for key in models.REGISTRY:
    spec = models.REGISTRY[key]
    state = "on this machine" if models.is_cached(key) else "not downloaded"
    print(f"  {spec.label:<28} {spec.params:>5}   {state}")

print()
print("  Downloading one is a single line:")
print("     model, tok = llm.load_local('smol')")

  Qwen3.5 0.8B                  0.8B   on this machine
  Qwen3.5 4B                      4B   on this machine
  SmolLM3 3B                      3B   on this machine
  Microsoft Phi-4 mini          3.8B   on this machine
  Google Gemma 4 E2B            2.3B   on this machine
  NVIDIA Nemotron 3 Nano 4B       4B   on this machine

     model, tok = llm.load_local('smol')


## 8. 🧪 Ask this model something of your own

The tools take SQL, so anything you can describe with a `WHERE` clause works.
Change the sentence and run it.

In [11]:
your_question = "Which five parcels in Hilldale have the largest lots?"

mine = agent.ask(assistant, your_question, system=prompts.REVIEW_ASSISTANT)
viz.show_trace(mine["trace"], answer=agent.final_answer(mine))

---

### 🕰️ *"Where we're going, we don't need a data centre."* 🕰️

---

**The take-home:** every notebook in this repository runs on hardware you
already own. Nothing in it requires a subscription, an internet connection, or
sending a single parcel record to anybody.